In [ ]:
RENDER_ENV = False
Render_Frame_rate=4
RESIZE_ENV = False
new_size = (56,80)
batch_size = 32
num_episodes = 43200
max_episode_steps = 100
num_stacked_frames = 4
num_frame_skip = 2

In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

no hay env para cerrar


In [ ]:
def calc_max_height(mat):
  mat = np.
  height = 20
  height_out =
  for row in mat:
    for col in row:
      if col > 0:
        return(height)
    height-=1
  return 0
def calc_holes(mat):
  mat = np.rot90(mat)
  holes = 0
  for row in mat:
    flag = False
    for col in row:
      if col > 0:
        flag = True
      elif flag:
        holes+=1
  return holes

SyntaxError: invalid syntax (ipython-input-1141065167.py, line 2)

In [ ]:
import cv2
import gymnasium as gym
import numpy as np
from tetris_gymnasium.envs.tetris import Tetris
from tetris_gymnasium.wrappers.observation import RgbObservation, FeatureVectorObservation
from tetris_gymnasium.wrappers.grouped import GroupedActionsObservations
from gymnasium.wrappers import TimeLimit, ResizeObservation, RecordVideo, MaxAndSkipObservation
from stable_baselines3 import DQN, PPO
from collections import deque

class CustomRewardWrapper(gym.RewardWrapper):
    """
    Custom reward shaping to encourage forward movement.
    This wrapper modifies the reward based on the agent's horizontal position.
    """
    def __init__(self, env, holes_penalty=-0.005, heigh_penalty=-0.2):
        super(CustomRewardWrapper, self).__init__(env)
        self.holes_penalty = holes_penalty
        self.heigh_penalty = heigh_penalty

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        game_variables = env.unwrapped.get_state().board[:-4, 4:-4]
        self.previous_max_height = calc_max_height(game_variables)

        return obs, info

    def reward(self, reward):
        #print(f"Reward original: {reward}")
        # Probar mayor penalizacion de agujeros
        custom_reward = reward
        game_variables = env.unwrapped.get_state().board[:-4, 4:-4]

        if game_variables.any():
            current_max_height = calc_max_height(game_variables)
            current_holes = calc_holes(game_variables)
            if current_max_height > self.previous_max_height:
              custom_reward+=self.heigh_penalty
            custom_reward += current_holes*self.holes_penalty
        return custom_reward

if __name__ == "__main__":
    env = gym.make("tetris_gymnasium/Tetris", render_mode="rgb_array")
    env = FeatureVectorObservation(env)
    #env = RgbObservation(env)
    #env = GroupedActionsObservations(env)
    #env = FrameStackObservation(env, stack_size=num_stacked_frames)
    env = CustomRewardWrapper(env)
    env.reset(seed=42)
    #model = DQN("MlpPolicy", env, verbose=1, buffer_size=10000) # para Mlp usar FeatureVectorObservation para Cnn
    model = PPO("MlpPolicy", env, verbose=1) # para Mlp usar FeatureVectorObservation para Cnn
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save("/content/drive/MyDrive/Models_Saves/PPO_tetris_1")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 36.9       |
|    ep_rew_mean          | 0.559      |
| time/                   |            |
|    fps                  | 230        |
|    iterations           | 4          |
|    time_elapsed         | 35         |
|    total_timesteps      | 8192       |
| train/                  |            |
|    approx_kl            | 0.01936686 |
|    clip_fraction        | 0.237      |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.98      |
|    explained_variance   | 0.0918     |
|    learning_rate        | 0.0003     |
|    loss                 | 0.464      |
|    n_updates            | 30         |
|    policy_gradient_loss | -0.0396    |
|    value_loss           | 0.967      |
----------------------------------------
-----------------------------------------
| r

KeyboardInterrupt: 

In [ ]:
try:
  env = RecordVideo(
    env,
    video_folder='/content/drive/MyDrive/Video_IA',    # Folder to save videos
    name_prefix="eval",               # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
  )
except:
  print('error implementando grabacion')

In [ ]:
for episode in range(10):
  state, info = env.reset()
  total_reward = 0
  done = False
  step_count = 0
  while not done:
    step_count+=1
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
  print(f"Episode: {episode} Reward: {total_reward} Steps: {step_count}")

In [ ]:
# var = len(env.unwrapped.get_state().board)
# count = var
# temp = env.unwrapped.get_state().board[:-4, 4:-4]
# print(temp)
# print(calc_max_height(temp))
# print(calc_holes(temp))
# #print(len(env.unwrapped.get_state().board))